# riverWQ EDA — Predicting E. coli Contamination in NZ Rivers

## Project Introduction

This notebook covers the exploratory data analysis required before the final modelling begins. The findings and conclusions will be used for preprocessing and modelling in '02_modelling.ipyb'.

# 1. Setup & Data Loading

## 1.1 Library Import

I will import the following libraries for EDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency


## 1.2 Dataset Loading 

In the following code cell, main dataset will be loaded.

In [ ]:
# Load the main dataset
df = pd.read_csv('../data/riverWQ_dataset.csv')


# 2. Initial Profiling

In this section, basic knowledge of the dataset will be worked out. And the quality and potential problems of the dataset will be monitored.

## 2.1 Dataset preview

In the following code cell, the basic information and overlook of dataset will be applied.

In [ ]:
# Show the shape of the dataset
print(f'Dataset shape: {df.shape}')
# Show the first 5 rows
df.head().T

## 2.2 Data types 

In this section, data types of the dataset will be shown to monitor the potential mismatch problems.

In [ ]:
# Show the dtype of each column
df.dtypes

**Data type observations**

All dtypes are stored properly, and no conversions are needed at this stage.

## 2.3 Missing value observation

In this section, missing values will be observed in the dataset.

In [ ]:
df.isnull().sum()

**Missing value findings**

13 columns have missing values and most of them show up in pairs('band' and 'median') apart from 'nitrate_band':

- 1. Target variables:
'ecolin_median' and 'ecoli_band' are both target variables, and each of them have 57 missing values. These rows can't be used for training or evaluation. 

- 2. Other columns:
Other columns that have missing values such as 'no3n_band' and so on represent water quality outcomes measured alongside E. coli, so they will be excluded later from modelling to avoid data leakage. Thus, no more actions are needed.

## 2.4 Duplicates

In this section, I will check whether the duplicates exist in the dataset.

In [ ]:
# Count exact duplicate rows
df.duplicated().sum()

**Observations**

There are no duplicates being observed in the dataset.

## 2.5 Implausible values

In this section, numerical values will be checked through to identify any implausible values or unusual values.

In [ ]:
# Summary statistics for all numerical columns
df.describe().T

In [ ]:
# Verify the two columns are truly identical
(df['pct_cropland'] == df['pct_cropping_horticulture']).all()

**Observations**

No implausible or impossible values are identified across all attributes in the dataset.

While 'pct_cropland' and 'pct_cropping_horticulture' have identical summary statistics (same mean, std, min, max). And verification confirms that they contain identical values. This finding needs discussing furtherly later on.

# 3. Univariate Analysis

This section works out the distribution of each feature and both target variables. The findings in this section will inform preprocessing and modelling decisions.

Two targets variables will be conducted first, followed by numerical and categorical features.

Some attributes should be excluded from univariate analysis.

'site_id', 'catchment', 'latitude' and 'longitude'are identifiers that can be excluded. 

Other water quality attributes such as 'drp_median', 'clarity_median', 'ammoniacal_n_median', 'no3n_median', 'ton_median' and their band will be excluded of analysis here since they do not represent catchment-level characteristics. And the project aims to assess how well catchment-level data can explain E. coli contamination.


## 3.1 Target variables: 'ecoli_median' and 'ecoli_band'

The two target variables are the most important variables in this dataset. So in the following section, statistics, visualisations and analysis on each target variable will be conducted.

### 3.1.1 'ecoli_median'(Regression target)

Its distribution, tendency and shape will be examined for analysis and discussion.

**The statistics will be conducted first**

In [ ]:
# Statistics for ecoli_median 
print('=== ecolin_median statistics ===')
print(df['ecoli_median'].describe())

# Skewness and kurtosis
print(f"Skewness: {df['ecoli_median'].skew():.3f}")
print(f"Kurtosis: {df['ecoli_median'].kurtosis():.3f}")

**Statisitcs interpretation**

Skewness is 3.046 which indicates an extreme right skew. And the kurtosis is 11.168 which means it has a long heavy tail. The maximum number also reflects this. The shape shows that most rivers have moderate E. coli while a small group produce extreme high contamination.

This distribution indicates that log transformation will be needed for modelling.

**Visualisations are below**

In [ ]:
# Create a fig and axes
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Use seaborn to draw histplots
sns.histplot(df['ecoli_median'].dropna(), bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribution of ecoli_median')
axes[0].set_xlabel('E. coli median (MPN per 100mL)')
# Use seaborn to draw boxplots
sns.boxplot(x=df['ecoli_median'].dropna(), ax=axes[1])
axes[1].set_title('Boxplot of ecoli_median')
axes[1].set_xlabel('E. coli median (MPN per 100mL)')
# Save the plots itto figures
plt.tight_layout()
plt.savefig('../figures/01_ecoli_median_raw.png', bbox_inches='tight')
plt.show()

**Visualisation findings**

From the histplot, it is clear that most data clustered at low values with a long tail extedning to around 2700 MPN/100ml, which matches the finding in statistics.

Log transformation should be needed for regression and the outliers in the boxplots should be the high-contamination situation but not the data errors.

**Will try to conduct log transformation to the dataset and see how visualisation looks**

In [ ]:
# Log transformation here and will be used later on too
df['log_ecoli_median'] = np.log1p(df['ecoli_median'])
# Create a fig and axes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Use seaborn to draw histplots
sns.histplot(df['log_ecoli_median'].dropna(), bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribution of log(ecoli_median)')
axes[0].set_xlabel('log(E. coli median)')
# Use seaborn to draw boxplots
sns.boxplot(x=df['log_ecoli_median'].dropna(), ax=axes[1])
axes[1].set_title('Boxplot of log(ecoli_median)')
axes[1].set_xlabel('log(E. coli median)')
# Save plots into figures
plt.tight_layout()
plt.savefig('../figures/02_ecoli_median_log.png', bbox_inches='tight')
plt.show()
# Skewness after transformation
print(f"Skewness after log transformation: {df['log_ecoli_median'].skew():.3f}")

**Findings after the log transformation**

After the log transformation, the distribution is approximatel symmetric as is shown above. And skewness dropped from 3.046 to near 0. Thus, the log transformation is effective and the regression target will be modelled after log transformation.

### 3.1.2 'ecoli_band'(Classification target)

'ecoli_band' is a attribute band derived from 'ecoli_median' using fixed thresholds. The class distribution will be examined first.

**Frequency and percentage table will be calculated first**

In [ ]:
# Frequency and percentage of each band
band_counts = df['ecoli_band'].value_counts().sort_index()
band_pct = df['ecoli_band'].value_counts(normalize=True).sort_index() * 100

# Combine into a clean table
band_summary = pd.DataFrame({
    'Count': band_counts,
    'Percentage': band_pct.round(1)
})
print(band_summary)

**A bar chart will be drawn below to show the visualisation.**

In [ ]:
# Create a fig and set the size
plt.figure(figsize=(8, 5))
sns.countplot(x=df['ecoli_band'], order=['A', 'B', 'C', 'D', 'E'])

# Title and axis labels
plt.title('Distribution of ecoli_band')
plt.xlabel('NPS-FM 2020 Attribute Band')
plt.ylabel('Number of sites')

# Save figure
plt.tight_layout()
plt.savefig('../figures/03_ecoli_band_distribution.png', bbox_inches='tight')
plt.show()

**Findings**

- 1. Barchart analysis

The classification target ecoli_band is unevenly distributed across the five bands. Bands D (104) and E (101) dominate, together accounting for over half the sites, while bands B (33) and C (20) are sparsely populated. Band A has 81 sites.

This is a class imbalance issue. The minority bands (B and C) have few samples, which may make them harder for a classification model to learn and predict reliably. This will need to be considered during modelling.

- 2. Inconsistency 

The provided 'ecoli_band' does not align exactly with the NPS-FM 2020 median thresholds when applied to 'ecoli_median' alone. This is not a data error. The official NPS-FM 2020 band is determined using several criteria together, including the median, the 95th percentile, and exceedance rates over 260 and 540 MPN/100mL, whereas this dataset provides only the median. The provided band therefore reflects information beyond what ecoli_median captures. 

he provided 'ecoli_band' will be used as the classification target, as it is the official band assigned by the monitoring authority and is what this project aims to predict.

## 3.2 Numerical features

There are 17 numerical features(16 'pct_' and 'mean_annual_rainfall_mm') that will be conducted univariate analysis here. All the numerical features will firstly be scanned in one plot, then specific statistics will be computed to further analysis.

**A list will be created which contains all the numerical features with 'pct_', making it easier for the following actions and codes.

In [ ]:
# Pick all columns that start with "pct_"
pct_cols = [c for c in df.columns if c.startswith('pct_')]

# Add rainfall to make the full list of numerical features
numerical_features = pct_cols + ['mean_annual_rainfall_mm']

print(numerical_features)

### 3.2.1 Distribution overview

A big plot showing the histogram of all 17 numerical features at once, which will provide a quick view of overall looks and shapes.

In [ ]:
# Plot histograms for all 17 numerical features in one grid
df[numerical_features].hist(figsize=(20, 30), bins=30)

# Add an overall title
plt.suptitle('Distributions of All Numerical Features', fontsize=16, y=1.00)

plt.tight_layout()
plt.savefig('../figures/04_numerical_distributions_grid.png', bbox_inches='tight')
plt.show()

**Initial observations(based on the histogram)**

Over half of the 17 numerical features are right-skewed with many zeros. And a few features appear bimodal or multimodal ('pct_forest', 'pct_exotic_grassland' 'pct_grassland_other_herbaceous_vegetation', 'pct_exotic_forest') 

### 3.2.2 Statistics

To quantify the visual patterns observed in 3.2.1, I will compute two key statistics for each numerical feature: skewness (shape) and the proportion of zero values (how often the feature is absent).

In [ ]:
# Calculate skewness and zero proportion for each numerical feature
skew_table = pd.DataFrame({
    'skewness': df[numerical_features].skew().round(2),
    'zero_pct': (df[numerical_features] == 0).mean().round(3) * 100
})

# Sort by skewness
skew_table = skew_table.sort_values('skewness', ascending=False)
skew_table

**Observations and implications**

- 1. Observations

Several variables show strong positive skewness, especially 'pct_exotic_scrub_shrubland', 'pct_cropland', 'pct_cropping_horticulture', 'pct_other_herbaceous_vegetation', 'pct_artificial_bare_surfaces', and 'pct_urban_area'. This means that most sites have low percentages of these land-cover types, while a small number of sites have much higher values.

The zero-value percentage shows how often a land-cover type is completely absent from a catchment. For example, 'pct_tussock_grassland', 'pct_artificial_bare_surfaces', and 'pct_cropland' have high zero percentages. These zeros are meaningful values rather than missing data, because they indicate that the land-cover type is not present in those catchments.

Some variables, such as 'pct_exotic_grassland', 'pct_grassland_other_herbaceous_vegetation', and 'pct_forest', have lower skewness and very few zero values. This suggests that these land-cover types are more commonly present across the monitoring sites.

- 2. Implications

Many pct features show strong positive skewness, but log transformation is not suitable here because log(0) is undefined and zeros are common across these columns. Tree-based models are naturally insensitive to skewness, while linear models and MLPs can handle it through standardisation. Therefore, pct features will be kept in their original form and standardised (e.g. StandardScaler) rather than log-transformed.
'mean_annual_rainfall_mm' has no zero values but operates on a very different scale to the percentage features. It will be scaled alongside the pct features.

## 3.3 Categorical features

There are 4 categorical features in the dataset which are 'region', 'wfs_land_cover', 'wfs_altitude', and 'rec_land_cover_type'. This section examines the frequency distribution of each to identify any class imbalance or rare categories.

In the following code cell, frequency and percentage table will be calculated first. And bar chart as a visualisation will be conducted.

In [ ]:
# Frequency and percentage of each categorical feature
cat_cols = ['region', 'wfs_land_cover', 'wfs_altitude', 'rec_land_cover_type']

for col in cat_cols:
    counts = df[col].value_counts()
    pct = (counts / len(df) * 100).round(1)
    print(f'\n--- {col} ---')
    print(pd.DataFrame({'count': counts, 'percent': pct}))

# Bar charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), cat_cols):
    df[col].value_counts().plot.bar(ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(f'Distribution of {col}')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../figures/05_categorical_distributions.png', bbox_inches='tight')
plt.show()

**Observations**

'region' has 12 categories with a highly uneven distribution and shows class imbalance. Otago(97) and Waikato(84) account for 45% of all. While Southland(2), Marlborough(3) and Tasman(7) have very few dites. These few categories may need to be handled during encoding and modelling.

'wfs_land_cover' has three classes(Rural, Forest, Urban) with Rural being the most common.

'wfs_altitude' is a binary split.

'rec_land_cover_type' is dominated by Pasture (252), followed by Native vegetation (110). Exotic forest has only 7 sites, the imbalance is similar to the rare region.

# 4. Bivariate Analysis

This section examines the relationships between features and each target variable, as well as the relationships among features themselves. Its aim is to identify which catchment characteristics are most strongly associated with E. coli contamination and to detect multicollinearity among features. These findings will directly inform feature selection and modelling decisions.

The analysis will be organised into three parts:

- 4.1 Features and regression target(ecoli_median)

- 4.2 Features and classification target(ecoli_band)

- 4.3 Features to feature relationships

## 4.1 Features and regression target(ecoli_median)

This section examines the relationship between numerical features & categorical features and 'ecoli_median'

### 4.1.1 Numerical feature and regression target(ecoli_median)

Spearman correlation will be used here for analysis instead of Pearson because lots of them are right-skewed and non-linear.

In [ ]:
# Spearman correlation between numerical features and log_ecoli_median
pct_cols = [c for c in df.columns if c.startswith('pct_')]
numerical_features = pct_cols + ['mean_annual_rainfall_mm']

corr_target = df[numerical_features + ['ecoli_median']].corr(method='spearman')['ecoli_median'].drop('ecoli_median').sort_values(ascending=False)

pd.DataFrame({'spearman_corr': corr_target.round(3)})

**Observations**

'pct_urban_area' shows the strongest positive correlation (0.42) with log(ecoli_median). pct_exotic_grassland (0.19) and pct_artificial_bare_surfaces (0.19) show weaker positive associations.

The strongest negative correlations are 'pct_natural_bare_lightly-vegetated_surfaces' (-0.57) and 'pct_tussock_grassland' (-0.49). 'pct_indigenous_scrub_shrubland' (-0.32) and 'pct_scrub_shrubland' (-0.29) also show negative associations.

'mean_annual_rainfall_mm' has a weak negative correlation (-0.18). 'pct_cropland' and several other features show near-zero correlation, suggesting minimal monotonic association.

The pattern is consistent with domain knowledge: contamination rises with urban and pastoral land cover, and falls in natural or undeveloped catchments (tussock, bare surfaces, native scrub). The strongest signals come from features describing the absence of human modification rather than its presence, which suggests these land cover features will be the most useful predictors. 

However, all correlations are modest, indicating that catchment land cover alone explains only part of the variation in E. coli — as expected for environmental data.

**To examine the form of the strongest relationships, scatter plots are produced for the four features most correlated with E. coli concentration: two positive (pct_urban_area, pct_exotic_grassland) and two negative (pct_natural_bare_lightly-vegetated_surfaces, pct_tussock_grassland). The log-transformed target is used so points are not compressed at the lower end.**

In [ ]:
# Scatter plots of the four strongest-correlated features vs log(ecoli_median)
key_feats = [
    'pct_urban_area',
    'pct_exotic_grassland',
    'pct_natural_bare_lightly-vegetated_surfaces',
    'pct_tussock_grassland'
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# Flat the axes for loop.
for ax, feat in zip(axes.flatten(), key_feats):
    ax.scatter(df[feat], df['log_ecoli_median'], alpha=0.4)
    ax.set_xlabel(f'{feat} (%)')
    ax.set_ylabel('log(ecoli_median)')
    ax.set_title(f'log(ecoli_median) vs {feat}')

plt.tight_layout()
plt.savefig('../figures/06_scatter_key_features_ecoli.png', bbox_inches='tight')
plt.show()

**Observations**

These scatter plots confirm the correlation results. For 'pct_urban_area', sites with any urban land tend to have higher E. coli, while clean sites only appear where urban cover is near zero.

For 'pct_exotic_grassland', higher pastoral cover does not always mean high E. coli, but very low (clean) values disappear as grassland increases.

For 'pct_natural_bare_lightly-vegetated_surfaces' and 'pct_tussock_grassland', sites with more of these natural land types tend to have lower E. coli.

None of the relationships is a straight line. The patterns are noisy, which suggests non-linear models may fit this data better than a simple linear model.

### 4.1.2 Categorical feature and regression target(ecoli_median)

The relationship between categorical features and E. coli concentration is examined using boxplots of log(ecoli_median) grouped by each category. This shows whether contamination levels differ systematically across land cover types, altitude classes, and regions.

In [ ]:
# Boxplots of log(ecoli_median) grouped by categorical features (apart from 'region' since it has many classes)
cat_feats = ['wfs_land_cover', 'rec_land_cover_type', 'wfs_altitude']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, col in zip(axes, cat_feats):
    order = df.groupby(col)['log_ecoli_median'].median().sort_values().index
    groups = [df[df[col] == cat]['log_ecoli_median'].dropna() for cat in order]
    ax.boxplot(groups, tick_labels=order)
    ax.set_title(f'log(ecoli_median) by {col}')
    ax.set_ylabel('log(ecoli_median)')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../figures/07_boxplot_ecoli_by_categorical(apart from region).png', bbox_inches='tight')
plt.show()

**Observations**

All three categorical features show a clear relationship with E. coli levels.

For 'wfs_land_cover', contamination increases from Forest (lowest) to Rural to Urban (highest). The gap between Forest and Urban is large.

'rec_land_cover_type' shows the same trend with more detail: Native vegetation is the cleanest, followed by Exotic forest, then Pasture, and Urban is the dirtiest. The four categories separate the sites more clearly than the three-category version. Pasture also shows several outliers with unusually low E. coli (log values around 1.5–2.5). 

For 'wfs_altitude', lowland sites have higher E. coli than upland sites. This makes sense, as lowland areas tend to have more farming and urban development.

These categorical features separate E. coli levels more clearly than most numerical features. 'rec_land_cover_type' looks the most useful because it has finer categories. All three need encoding (e.g. one-hot) before modelling. Since 'rec_land_cover_type' and 'wfs_land_cover' describe similar things, they may be correlated — this will be checked in Section 4.3.

**Boxplots of log(ecoli_median) grouped by 'region' will be presented below since it has many classes for better presence.

In [ ]:
# Boxplots of log(ecoli_median) grouped by 'region' 
order = df.groupby('region')['log_ecoli_median'].median().sort_values().index
groups = [df[df['region'] == r]['log_ecoli_median'].dropna() for r in order]

fig,ax = plt.subplots(figsize = (10,7))
ax.boxplot(groups, tick_labels=order, vert=False)
ax.set_title(f'log(ecoli_median) by region')
ax.set_ylabel('log(ecoli_median)')

plt.tight_layout()
plt.savefig('../figures/08_boxplot_ecoli_by_region.png', bbox_inches='tight')
plt.show()

**Observations**

E. coli levels differ a lot between regions. Wellington, Auckland, Taranaki, and Waikato have the highest medians (above 5). Marlborough, Hawke's Bay, and Tasman are the lowest (below 4). Otago sits in the middle even though it has the most sites (97).

The pattern likely reflects land use and climate — heavily farmed or urban regions tend to be more contaminated, while drier or less developed regions are cleaner.

Some regions have very few sites (Southland 2, Marlborough 3, Tasman 7). Their boxplots are based on too few points to be reliable — one extra site could shift the median a lot. This matches the class imbalance noted in Section 3.3.

## 4.2 Features and classification target(ecoli_band)

This section examines the relationship between numerical features & categorical features and 'ecoli_band'.

### 4.2.1 Numerical and 'ecoli_band'

Since it is mentioned before that 'ecoli_median' might not be the only factor that affects 'ecoli_band'. To examine and verify how numerical features vary across the five quality bands, the median of each feature is calculated for each band. The gap between band A (cleanest) and band E (most contaminated) is used to identify which features differ most across band levels.

In [ ]:
# Median of each numerical feature by band
pct_cols = [c for c in df.columns if c.startswith('pct_')]
num_features = pct_cols + ['mean_annual_rainfall_mm']

band_medians = df.groupby('ecoli_band')[num_features].median().T
band_medians.columns = ['A', 'B', 'C', 'D', 'E']
band_medians['E_minus_A'] = (band_medians['E'] - band_medians['A']).round(2)
band_medians = band_medians.sort_values('E_minus_A', key=abs, ascending=False)

band_medians.round(2)

**Observations**

Across all numerical features, the four with the strongest band-separating patterns are 'pct_tussock_grassland', 'pct_natural_bare_lightly-vegetated_surfaces', 'pct_urban_area', and 'pct_exotic_grassland'. These are also the top features identified by Spearman correlation in Section 4.1.

The consistency between the two analyses indicates that 'ecoli_median' is still the main factor driving the band, even though the official band also includes 95th percentile and exceedance rates. These four features will be examined in more detail below.

The boxplots of the features selected will be worked out in the following code cell.

In [ ]:
# Boxplots of top features by band
top_feats = [
    'pct_tussock_grassland',
    'pct_urban_area',
    'pct_natural_bare_lightly-vegetated_surfaces',
    'pct_exotic_grassland'
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
band_order = ['A', 'B', 'C', 'D', 'E']

for ax, feat in zip(axes.flatten(), top_feats):
    groups = [df[df['ecoli_band'] == b][feat].dropna() for b in band_order]
    ax.boxplot(groups, tick_labels=band_order)
    ax.set_title(f'{feat} by ecoli_band')
    ax.set_xlabel('NPS-FM Band')
    ax.set_ylabel(f'{feat} (%)')

plt.tight_layout()
plt.savefig('../figures/09_numerical_features_by_band.png', bbox_inches='tight')
plt.show()

**Observations**

'pct_tussock_grassland': A and B sites have a wide range with many high-tussock catchments, while D and E sites are nearly all near zero. Clear monotonic decline.

'pct_urban_area': bands A–D are tightly near zero. Only band E shows a wide spread with extreme outliers (60–100%). Urban land flags the most contaminated band specifically but not a gradual indicator.

'pct_natural_bare': gradual decline from A/B to E. Less sharp than tussock but consistent.

'pct_exotic_grassland': only a weak upward shift across bands, with heavy overlap. A noisy signal at the band level.

The strongest band-separating features identify clean catchments ('pct_tussock_grassland', 'pct_natural_bare'). 'pct_urban_area' mainly flags band E. The classifier may find it easier to identify very clean (A/B) and very contaminated (E) sites than to distinguish the middle bands.

### 4.2.2 Categorical and 'ecoli_band'

For categorical features, the relationship with band is examined using crosstabs showing how each category is distributed across the five bands. Row percentages are used to make the comparison fair across categories of very different sizes.

The crosstabs of the categorical features(apart from region) by 'ecoli_band' will be conducted.

In [ ]:
# Crosstabs of each categorical feature against ecoli_band (row percentages)
cat_cols = ['wfs_land_cover', 'rec_land_cover_type', 'wfs_altitude']
band_order = ['A', 'B', 'C', 'D', 'E']

for col in cat_cols:
    ct = pd.crosstab(df[col], df['ecoli_band'])[band_order]
    ct_pct = (ct.div(ct.sum(axis=1), axis=0) * 100).round(1)
    print(f'\n--- {col} vs ecoli_band (row %) ---')
    print(ct_pct)

Heatmaps as the visualisation will be worked out.

In [ ]:
# Heatmaps of categorical features vs ecoli_band (row %)
cat_cols = ['wfs_land_cover', 'rec_land_cover_type', 'wfs_altitude']
band_order = ['A', 'B', 'C', 'D', 'E']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, cat_cols):
    sns.heatmap(ct_pct, annot=True, fmt='.1f', cmap='YlOrRd', cbar=False, ax=ax)
    ax.set_title(f'{col} vs ecoli_band (row %)')
    ax.set_xlabel('NPS-FM Band')
    ax.set_ylabel(col)

plt.tight_layout()
plt.savefig('../figures/10_categorical_features_by_band.png', bbox_inches='tight')
plt.show()

**Observation**

'wfs_land_cover': Forest sites are concentrated in band A (50%) and rarely fall into C(5.1%) and E (10%). Urban sites show the opposite — 0% are in A/B/C, 25% in D, and 75% in E. Rural sites are spread across all bands but lean toward D and E (combined ~69%).

'rec_land_cover_type': Native vegetation sites are dominated by A (55%) with only 3% in E. Pasture sites are heavily skewed to D and E (~75% combined). Urban sites are even more extreme — 80% are in band E. Exotic forest has only 6 sites total, too few to interpret reliably.

'wfs_altitude': Upland sites are heavily concentrated in band A (49%) with only 8% in E. Lowland sites are the opposite — 78% fall into D or E.

All three categorical features show strong, clear separation across bands. Land cover (especially 'rec_land_cover_type') is the strongest categorical predictor. Altitude provides a similarly strong split between upland (cleaner) and lowland (dirtier) sites. These features will be useful for the classification model.

In [ ]:
# Crosstabs of 'region' against ecoli_band (row percentages)
band_order = ['A', 'B', 'C', 'D', 'E']


ct = pd.crosstab(df['region'], df['ecoli_band'])[band_order]
ct_pct = (ct.div(ct.sum(axis=1), axis=0) * 100).round(1)
print(f'\n--- region vs ecoli_band (row %) ---')
print(ct_pct)

In [ ]:
# Heatmaps of region vs ecoli_band (row %)
band_order = ['A', 'B', 'C', 'D', 'E']

fig, ax = plt.subplots(figsize=(8, 7))

ct_pct = ct_pct.sort_values('E', ascending=False)
sns.heatmap(ct_pct, annot=True, fmt='.1f', cmap='YlOrRd', cbar=False, ax=ax)
ax.set_title('region vs ecoli_band (row %)')
ax.set_xlabel('NPS-FM Band')
ax.set_ylabel('region')

plt.tight_layout()
plt.savefig('../figures/11_region_features_by_band.png', bbox_inches='tight')
plt.show()

**Observations**

Region carries information about contamination, but much of it overlaps with land cover and climate features already in the dataset. The small-sample regions also risk introducing spurious patterns if region is used as a model input. The decision on whether to include region as a feature will be revisited in the plan. Regardless, region will be used in the evaluation stage to break down model performance.

## 4.3 Features and features

This section examines relationships between features themselves, with two goals: identify pairs of features that carry similar information (multicollinearity), and decide whether to keep the broad-level or medium-level land cover percentages.

The findings here will directly inform feature selection in the preprocessing plan.

### 4.3.1 Numerical & numerical features

The spearman correlation matrix as well as the heatmap will be calculated and conducted first and whose goal is to identify pairs that carry overlapping information.

In [ ]:
# Spearman correlation matrix among numerical features
pct_cols = [c for c in df.columns if c.startswith('pct_')]
num_features = pct_cols + ['mean_annual_rainfall_mm']
# Calculate the spearman maxtrix
corr_matrix = df[num_features].corr(method='spearman')

# Heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Spearman correlation between numerical features')
plt.tight_layout()
plt.savefig('../figures/12_numerical_feature_correlation.png', bbox_inches='tight')
plt.show()

**Observations**

Several pairs of features show very strong correlations:

'pct_cropland' and 'pct_cropping_horticulture': ρ = 1.0 (already identified in univariate analysis — they are duplicates).
'pct_forest' and 'pct_indigenous_forest': ρ = 0.91.
'pct_scrub_shrubland' and 'pct_indigenous_scrub_shrubland': ρ = 0.91.
'pct_grassland_other_herbaceous_vegetation' and 'pct_exotic_grassland': ρ = 0.88.
'pct_tussock_grassland' and 'pct_natural_bare_lightly-vegetated_surfaces': ρ = 0.72.

The first four pairs each link a broad-level feature to its dominant medium-level sub-category. When a broad category is mostly made up of one medium sub-class (e.g. forest is mostly indigenous forest in most catchments), the two columns track each other almost perfectly.

There are also moderate negative correlations between broad land cover types (e.g. forest vs grassland: ρ = -0.71), reflecting that a catchment can only have so much land — if grassland is high, forest must be low.

'pct_indigenous_forest' and 'mean_annual_rainfall_mm' are positively correlated (ρ = 0.71), meaning that high-rainfall areas in New Zealand tend to be native forest catchments.

Keeping both broad and medium pct features together would introduce strong multicollinearity, which is harmful for linear models and uninformative for tree-based models. So more discussions and decisions will be needed.

### 4.3.2 Broad vs Medium features

The 16 pct features come in two levels: 5 broad categories and 11 medium sub-categories. Section 4.3.1 confirmed they are highly correlated, so using both would introduce strong multicollinearity. A choice must be made between the two levels.

Section 4.1 already calculated the Spearman correlation between each numerical feature and 'log_ecoli_median'. Those values are reused here, reorganised so each broad feature is shown alongside the medium sub-categories it contains. This makes it possible to see whether the medium level captures signals the broad level hides.

- 1. pct_cropland (+0.028)

    pct_cropping_horticulture: +0.028


- 2. pct_forest (-0.070)

    pct_indigenous_forest: -0.087
    pct_exotic_forest: +0.163

- 3. pct_grassland_other_herbaceous_vegetation (+0.011)

    pct_exotic_grassland: +0.192
    pct_tussock_grassland: -0.490
    pct_other_herbaceous_vegetation: -0.071

- 4. pct_scrub_shrubland (-0.294)

    pct_indigenous_scrub_shrubland: -0.323
    pct_exotic_scrub_shrubland: -0.057

- 5. pct_urban_bare_lightly-vegetated_surfaces (-0.069)

    pct_urban_area: +0.419
    pct_natural_bare_lightly-vegetated_surfaces: -0.568
    pct_artificial_bare_surfaces: +0.185

**Implications**

Use the medium pct features and drop the broad pct features. All the strongest signals in the dataset are at the medium level, while the broad features cancel out opposite-direction sub-signals and lose important information. Keeping both levels would also cause heavy multicollinearity. 'pct_cropping_horticulture' will be kept and 'pct_cropland' dropped, as they are identical (ρ = 1.0).

### 4.3.3 Categorical vs categorical features

The dataset contains two land cover categorical features that describe the quite similar information at different levels of detail:

'wfs_land_cover' has 3 categories: Forest, Rural, Urban.
'rec_land_cover_type' has 4 categories: Native vegetation, Exotic forest, Pasture, Urban.

To check whether they overlap, Cramér's V is calculated and decisions will be made.

In [ ]:
# Define a funciton called cramers_v
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

print(pd.crosstab(df['wfs_land_cover'], df['rec_land_cover_type']))
print(f"\nCramer's V: {cramers_v(df['wfs_land_cover'], df['rec_land_cover_type']):.3f}")

**Observations**

The two features show a strong association (Cramér's V = 0.697). The crosstab confirms a clear correspondence:

Forest sites are mostly Native vegetation (80/115). Rural sites are almost entirely Pasture (213/247). Urban sites in 'wfs_land_cover' map mostly to Urban in 'rec_land_cover_type' (24/34).

Only one of the two will be used. 'rec_land_cover_type' will be kept, as the four-category version provides finer band separation (Section 4.2.2) without losing the underlying signal. This decision will be revisited during modelling if the classifier performance is unexpectedly low.

# 5. Preprocessing and Modelling Plan

This section aims to answer the required questions and address the preprocessing and modelling plans based on the analysis above.

## 5.1 Feature selection and exclusion

- Features that will be excluded

1. Identifiers

'site_id', 'catchment', 'latitude', and 'longitude' carry no predictive content and risk overfitting to specific locations that would not generalise to regions outside the dataset (e.g. Canterbury).

2. Other water quality outcomes

'drp_median', 'clarity_median', 'ammoniacal_n_median', 'no3n_median', 'ton_median', and their corresponding bands are excluded. They are measured at the same sites as E. coli, not catchment-level predictors.

3. Duplicate feature

'pct_cropland'

4. Broad-level land cover features

'pct_forest', 'pct_grassland_other_herbaceous_vegetation', 'pct_scrub_shrubland', and 'pct_urban_bare_lightly-vegetated_surfaces' are dropped in favour of their medium-level sub-categories. 

5. 'wfs_land_cover'

6. 'region'

Excluded as a model input to avoid learning region-specific patterns that cannot transfer to unseen regions (e.g. Canterbury).

- Features that will be used

1. Medium-level pct features

pct_cropping_horticulture, pct_indigenous_forest, pct_exotic_forest, pct_exotic_grassland, pct_tussock_grassland, pct_other_herbaceous_vegetation, pct_indigenous_scrub_shrubland, pct_exotic_scrub_shrubland, pct_urban_area, pct_natural_bare_lightly-vegetated_surfaces, and pct_artificial_bare_surfaces. 

These carry the strongest signals identified in Sections 4.1 and 4.3.2. They will be standardised (e.g. StandardScaler) so they sit on a comparable scale for linear models and MLPs. Log transformation is not used because of the high proportion of true zeros (Section 3.2).

2. 'mean_annual_rainfall_mm'

Numerical feature with no zeros but a different scale from the pct features. Standardised alongside them.

3. 'wfs_altitude'

Binary categorical (lowland/upland) with a clean separation of E. coli levels (Section 4.1, 4.2.2). One-hot encoded.

4. 'rec_land_cover_type'

Four-category land cover feature with the strongest band separation of all categorical features (Section 4.2.2). One-hot encoded.

Target features:

'ecoli_median' — Apply log transformation (log1p) before modelling due to severe right skew (Section 3.1.1).

'ecoli_band' — Used as provided, since it is the official NPS-FM 2020 band that the project aims to predict (Section 3.1.2).

## 5.2 Data quality issues

1. Missing values in targets and co-measured other water quality columns.

57 rows have all five median-and-band column pairs missing simultaneously (Section 2). These rows cannot be used for either modelling task and will be dropped. After this, the working dataset is 339 rows.

2. Duplicate column. 

'pct_cropland' will be dropped; only the medium-level version is retained.

3. Mismatch between 'ecoli_band' and 'ecoli_median'

Strictly speaking, the NPS-FM 2020 'ecoli_band' is determined by four criteria together — the median concentration, 95th percentile, and exceedance rates over 260 and 540 MPN/100mL — but the dataset provides only the median. 

However, the analysis in Section 4.2.1 showed that the features most strongly separating the bands are the same features most correlated with ecoli_median, indicating that the median is still the main driver of the band even though the additional criteria also contribute. 

## 5.3 Algorithm selection

Three algorithms will be considered for each task. Each set includes a simple interpretable baseline and a Multi-layer Perceptron as required, and one additional algorithm suited to the dataset.

**Regression task (predicting ecoli_median)**

1. Linear Regression(baseline)

It is a simple starting point. The standardised numerical features and one-hot encoded categorical features are directly usable. Its limitation is that it assumes a linear relationship while Section 4.1 found that this dataset is not likely to be a linear one. But it can still be used and its result can be compared and analysed.

2. Random Forest Regressor

A tree-based ensemble that handles non-linear relationships, mixed feature types and skewed inputs without any requiring transformation. The threshold-like patterns observed in 4.1 are what tree-based is good at.

3. Multi-layer Perceptron(MLP) Regressor

As is required by the assignment. It is capable of learning non-linear interactions beteen features. The dataset is small, so a small architecture with regularisation will be needed to avoid overfitting.

**Classification task (predicting ecoli_band)**

1. Logistic Regression (baseline). 

It provides the required interpretable benchmark. Section 4.2 showed that the strongest signals (especially pct_urban_area and rec_land_cover_type) are threshold-like rather than smooth gradients across bands, so linear decision boundaries are expected to underperform — quantifying how much will be informative.

2. Random Forest Classifier. 

It should be particularly suitable for this task because the strongest band-separating features identified in Section 4.2 work through thresholds and category-band mappings (e.g. Urban → 80% in band E, Pasture → 75% in D/E, high tussock → A/B). Trees encode these rules naturally. It also handles the class imbalance (D/E dominate, B/C sparse) better than linear models, especially with class weighting.

3. Multi-layer Perceptron (MLP) Classifier. 

It can capture non-linear class boundaries. Section 3.1.2 showed band counts of 81/33/20/104/101 — the minority bands B and C will need to be handled through class weighting or stratified sampling during training and evaluated separately

## 5.4 Model framing justification

The classification task is expected to be more challenging than the regression task on this dataset, based on findings from the EDA.

1. The classification target carries information the model inputs cannot capture. Compared with 'ecoli_median', 'ecoli_band' is determined by four criteria together as discussed above. The dataset provides only the median although it has been tested that 'ecoli_median' is still the main driven of band level. 'ecoli_median' has no such issue to worry about.

2. Severe class imbalance. Bands D (104) and E (101) dominate while B (33) and C (20) are very few. Such minority group makes it harder for classifier to learn from. Regression treats every sample equally and is not affected in the same way.

3. Middle bands show little feature-level separation. Section 4.2.1 found that the strongest band-separating features (e.g. pct_urban_area) only differ sharply at band E, with bands A–D tightly clustered. This might mean the classifier has limited signal to distinguish neighbouring bands. 

## 5.5 Predictive features

Based on the EDA, four features stand out as the strongest predictors of E. coli contamination. 

1. 'pct_natural_bare_lightly-vegetated_surfaces' 

It is the single strongest signal in the dataset. It shows the strongest Spearman correlation with log_ecoli_median (ρ = -0.57, Section 4.1) and a clear decline in median value across bands: from around 1.5% in bands A/B to near zero in D/E (Section 4.2.1). This feature indicates undeveloped, unmodified catchments, which are typically associated with the cleanest water.

2. 'pct_tussock_grassland' 

It is another strong negative predictor. Section 4.1 found Spearman ρ = -0.49 with log_ecoli_median, and in Section 4.2.1, it shows that A/B around 6%, C dropping to 5%, D/E near zero. Tussock grassland indicates high-altitude native vegetation in catchments far from human activity.

3. 'pct_urban_area' 

It is the strongest positive numerical signal. Section 4.1 found Spearman ρ = +0.42 with log_ecoli_median, and the scatter plot showed that even small amounts of urban land almost guarantee elevated E. coli. The pattern is different in the band analysis — values stay near 0.3% across bands A–D and jump sharply to 1.4% in band E (Section 4.2.1), so it should act as a specific indicator for the most contaminated sites rather than a gradual one.

4. 'rec_land_cover_type' 

This one is the strongest categorical predictor. Section 4.1 found a clear ordering of medians from Native vegetation (cleanest) through Exotic forest and Pasture to Urban (dirtiest), with the extremes separated by over 3 log units. Section 4.2.2 showed the cleanest band separation of any feature. Its four category provides finer separation than wfs_land_cover.

**Observations**

The strongest signals come from features describing the absence of human modification more than from features describing its presence.Thus, it is easier to identify a clean catchment from what it contains (untouched land) than to identify a contaminated catchment from any single factor, since contamination can come from multiple sources that no individual feature fully captures.